# LightGBM v2 데드볼 보정

DFL 7경기에서 중단 시작과 재개 오류 여섯 가지를 비교한다. BAR–ATH 영상에 사용한 규칙 결합도 같은 노트북에서 적용할 수 있다.

[입력 준비, 실행 방법, 결과 해석](README_lightgbm_corrections.md)

## 1. 공통 설정

기존 LightGBM v2 특징과 동일한 DFL 경기 목록, 25fps, 기준 전환 벌점을 사용한다.

In [ ]:
import hashlib
from importlib.metadata import version
import json
import os
from pathlib import Path
import subprocess
import sys
from tempfile import TemporaryDirectory
from time import perf_counter
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.ndimage import maximum_filter1d, minimum_filter1d, uniform_filter1d
from scipy.special import expit, logit

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/train_dfl.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("저장소 폴더 안에서 노트북을 열어 주세요.")
MODEL_DIR = ROOT / "model/영상추출/모델/dead_ball"
RESULTS = ROOT / "model/영상추출/추출csv/dead_ball"
NOTEBOOK = MODEL_DIR / "LightGBM_deadball_corrections.ipynb"
def sha(path):
    with Path(path).open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()

sys.path.insert(0, str(ROOT / "src"))
import compare_smoothing as scoring
from config import CONFIG
import dfl_labels
import dfl_load
import features
import final_dead_balls as fd
import simulate
import smoothing
import train_dfl

MATCHES = tuple(dfl_load.MATCHES)
FPS, SEED, THREADS = CONFIG["fps"], CONFIG["sim_seed"], 6
TRAIN_SEEDS = {match: SEED + i for i, match in enumerate(MATCHES)}
PENALTY, BIAS = (CONFIG["smooth_choice"][key] for key in ("penalty", "bias"))
assert BIAS == 0 and FPS == 25
STRESS = dict(missing_share=0.25, gap_mean_s=2.0, jitter_m=0.20, smooth_frames=8)
# 각 바깥 폴드에서 미리 정한 한 경기를 내부 검증에 사용한다.
INNER = {held: MATCHES[(i+1) % len(MATCHES)] for i, held in enumerate(MATCHES)}
VARIANTS = {"baseline": (), "1_start_late": (1,), "2_start_early": (2,),
            "3_alive_late": (3,), "4_alive_early": (4,), "5_false_dead": (5,),
            "6_short_missed": (6,), "rules_combined": (1, 2, 3, 4, 5, 6),
            "full_combined": (1, 2, 3, 4, 5, 6)}
SPLIT_VARIANTS = ("feature_contri", "forced_split")
RULE_GRID = (
    dict(margin=0.50, strength=.65, bonus=.45),
    dict(margin=0.25, strength=.65, bonus=.45),
    dict(margin=0.25, strength=1.0, bonus=.85),
    dict(margin=0.00, strength=1.0, bonus=.85),
)
FORCED_THRESHOLDS = (1.0, 1.75)
P75_WEIGHTS = (.55, .8)
BASELINE_F1 = {"alive_spell_f1": 0.8770, "dead_f1": 0.9459}
BASELINE_ERRORS = [113, 150, 137, 111, 43, 32]
COMPETITION_RULES = dict(margin=0.00, strength=1.0, bonus=.85)

## 2. 선수·공 특징과 DFL 입력

라인 교차, 패스·수신, 재개 후보를 추적 좌표에서 추정한다. 이벤트는 평가 시 오류 유형을 나누는 데 쓴다.

In [ ]:
def halves(period):
    cuts = np.r_[0, np.flatnonzero(period[1:] != period[:-1]) + 1, len(period)]
    return [slice(a, b) for a, b in zip(cuts[:-1], cuts[1:])]


def shift(x, frames):
    out = np.full(x.shape, np.nan, dtype=np.float64)
    if not frames:
        return np.asarray(x, dtype=np.float64)
    if 0 < frames < len(x):
        out[frames:] = x[:-frames]
    elif -len(x) < frames < 0:
        out[:frames] = x[-frames:]
    return out


def spread(mask, radius):
    return maximum_filter1d(mask.astype(np.uint8), 2 * radius + 1, mode="constant") != 0


def extend_from(indices, n, before, after):
    counts = np.zeros(n + 1, dtype=np.int32)
    np.add.at(counts, np.maximum(0, indices - before), 1)
    np.add.at(counts, np.minimum(n, indices + after + 1), -1)
    return np.cumsum(counts[:-1]) > 0


def peak_indices(mask, strength, gap):
    peaks = np.array([a + np.argmax(strength[a:b]) for a, b in features.runs_of(mask)], dtype=int)
    return peaks[np.r_[True, np.diff(peaks) >= gap]] if peaks.size else peaks


def ball_geometry(match, condition, seed):
    ball = match.ball.astype(np.float64, copy=True)
    if condition == "masked":
        ball[:] = np.nan
    if condition == "stress":
        for i, half in enumerate(halves(match.period)):
            rng = np.random.default_rng(seed + i)
            noisy = simulate.ball_dropout(ball[half], rng, STRESS["missing_share"], STRESS["gap_mean_s"])
            noisy += simulate.smooth_noise(len(noisy), 1, STRESS["jitter_m"], STRESS["smooth_frames"], rng)[:,0]
            ball[half] = noisy
    return ball


def movement_features(players, team, ball, pitch):
    n = len(ball)
    observed = np.isfinite(ball).all(axis=1)
    x,y = ball.T
    length,width = pitch
    distance = np.minimum.reduce((x, length-x, y, width-y))
    delta = ball - shift(ball, 1)
    jumps = np.linalg.norm(delta, axis=1) > 45 / FPS
    continuous = minimum_filter1d(observed.astype(np.uint8), 7, mode="constant").astype(bool)
    reliable = continuous & ~spread(jumps, 4)
    velocity = (shift(ball,-3) - shift(ball,3)) * FPS / 6
    speed = np.linalg.norm(velocity, axis=1)
    speed[~reliable] = np.nan
    dist = np.linalg.norm(players - ball[:,None,:],axis=2)
    dist = np.where(np.isfinite(dist), dist, np.inf)
    near = dist.min(axis=1)
    owner = np.argmin(dist,axis=1)
    old_owner = np.nan_to_num(shift(owner,4),nan=-1).astype(int)
    same = owner == old_owner
    rows = np.arange(n)
    dr = players[rows,owner] - shift(players,4)[rows,owner]
    ball_motion = ball - shift(ball,4)
    co_motion = np.linalg.norm(dr-ball_motion,axis=1)
    track_carry = reliable & same & (near < 1.1) & (co_motion < .40) & (speed > .35) & (speed < 7)
    carry = uniform_filter1d(track_carry.astype(float), 15, mode="constant") >= .65
    near_goal = (np.minimum(x,length-x) < 16.5) & (abs(y-width/2)<20.2)
    keeper_candidate = carry & near_goal
    pass_windows = np.zeros(n, dtype=bool)
    launch = np.nan_to_num(shift(speed,-5),nan=0)
    before = np.nan_to_num(shift(speed,5),nan=0)
    persistent = uniform_filter1d((speed>3.0).astype(float), 9, mode="constant")
    departure = (launch > 3.8) & (launch-before > 1.3) & (near < 2.0) & reliable & (persistent > .35)
    departures = peak_indices(departure,launch-before,10)
    # Find a release followed by a different player controlling the ball.
    for a in departures:
        b = min(n,a+51)
        window = np.arange(a+7,b)
        hits = window[reliable[window] & (near[window]<1.35) & (owner[window]!=owner[a])]
        if hits.size:
            h = int(hits[0])
            flight = ball[a:h+1]
            span = np.linalg.norm(ball[h]-ball[a])
            if (span >= 4 and np.isfinite(flight).all() and not jumps[a:h+1].any()
                    and np.min(distance[a:h+1]) >= 0
                    and (np.nanmedian(speed[max(a,h-4):min(n,h+5)]) < launch[a]*.8 or near[h]<.7)):
                pass_windows[a:h+1] = True
    side_guess = team[owner]
    team_left = np.array([np.nanmedian(players[:,team==k,0]) for k in (0,1)])
    home_left = team_left[0] < team_left[1]
    own_third = np.where((side_guess==0)==home_left, x<length/3, x>2*length/3)
    restart = departure & (np.nan_to_num(shift(persistent,-6),nan=0)>.48)
    throw = restart & (np.minimum(y,width-y)<2.5) & (shift(distance,-9)-distance>.6)
    kick = restart & ~throw
    restart_ids = peak_indices(restart, launch-before, 13)
    restart_at = np.zeros(n,dtype=bool)
    restart_at[restart_ids] = True
    # Relative change in the same player's motion. Missing identities never count as stopped.
    vv = (shift(players,-3)-shift(players,3)) * FPS/6
    ps = np.linalg.norm(vv,axis=2)
    pv = shift(vv,9)
    old_ps = np.linalg.norm(pv,axis=2)
    ds = ps - old_ps
    good = np.isfinite(ds) & (old_ps < 12) & (ps < 12)
    def share(flag, valid):
        return np.divide((flag & valid).sum(axis=1), valid.sum(axis=1),
                         out=np.zeros(n), where=valid.sum(axis=1)>0)
    team_decel = np.column_stack([share(ds[:,team==k]<-.35,good[:,team==k]) for k in (0,1)])
    synchronized = team_decel.min(axis=1)
    ball_interrupted = reliable & (before>3.0) & (launch < before*.50) & (near<3.0)
    # A stationary ball alone never activates this cue.
    stop_hint = ball_interrupted & (synchronized>.32) & (distance>1)
    last_index = np.maximum.accumulate(np.where(observed, rows, -1))
    last = ball[np.maximum(last_index,0)]
    near_last = np.linalg.norm(players-last[:,None,:],axis=2)<6
    change = np.linalg.norm(vv-pv,axis=2)
    local = share((ds>.4) | ((change>1.25)&(ps>.7)), near_last & good)
    fallback = ~reliable & (last_index>=0) & (rows-last_index<=FPS*2) & (local>.35)
    preparation = reliable & ~spread(restart_at,8) & ~pass_windows & ((near<2.2)&(persistent<.55) | spread(jumps,6))
    sig = {
        "reliable":reliable, "distance":distance, "speed":speed, "restart":restart_at,
        "kick":kick, "throw":throw, "pass":pass_windows, "carry":carry, "keeper":keeper_candidate,
        "back_pass":pass_windows & own_third, "prep":preparation, "stop":stop_hint,
        "fallback":fallback, "local":local, "decel":synchronized, "jumps":spread(jumps,6),
    }
    before_dist = shift(distance, 5)
    after_dist = shift(distance, -4)
    for margin in (0.0,.25,.50):
        crossing = reliable & (shift(distance,1)>=-margin) & (distance < -margin)
        crossing &= (before_dist-distance>.15) & (distance-after_dist>-.06)
        ids = peak_indices(crossing, np.nan_to_num(before_dist-distance,nan=0), 10)
        sig[f"cross_{margin}"] = ids
    return sig


def label_start_kinds(spells, events, timestamp):
    """오류 집계에만 쓰는 중단 유형. 같은 하프에서 시작 ±2초 이내의 이벤트를 연결한다."""
    types = {"FOUL_COMMITTED": "foul", "GENERIC:Offside": "offside", "BALL_OUT": "other_lineout"}
    candidates = events[events.event_type.isin(types) & (events.timestamp_s >= 0)]
    kinds = {}
    for half, group in spells.groupby("period"):
        nearby = candidates[candidates.period_id == half]
        times = nearby.timestamp_s.to_numpy()
        names = nearby.event_type.to_numpy()
        for spell in group.itertuples(index=False):
            kind = spell.cause
            if len(times):
                distances = np.abs(times - timestamp[spell.start])
                closest = int(distances.argmin())
                if distances[closest] <= 2:
                    event = names[closest]
                    if event != "BALL_OUT" or kind not in ("throw_in", "goal_kick", "corner"):
                        kind = types[event]
            kinds[int(spell.start)] = kind
    return kinds


def prepare_data():
    train_dfl.set_feature_set("v2")
    train, data = {}, {}
    columns = None
    for match_id in MATCHES:
        seed = TRAIN_SEEDS[match_id]
        sampled = train_dfl.match_features(match_id, "rand", seed).iloc[::CONFIG["train_stride"]]
        match_columns = train_dfl.model_columns(sampled.columns)
        if columns is None:
            columns = match_columns
            assert len(columns) == 264
        assert match_columns == columns, match_id
        train[match_id] = (sampled[columns].to_numpy(dtype=np.float32, copy=True),
                           sampled.dead.to_numpy(dtype=np.int8, copy=True))
        del sampled
        track, events, meta = dfl_load.load_match(match_id)
        match = features.match_from_dfl(track, meta)
        observed_players = simulate.simulate(match, seed=SEED).players
        frame = train_dfl.match_features(match_id, "sim", SEED)
        values = frame[columns].to_numpy(dtype=np.float32, copy=True)
        labels = frame[["dead", "period"]]
        del frame
        geometry = fd.ball_signals(match)
        crossings = fd.line_crossings(geometry)
        spells = dfl_labels.dead_spells(track, events)
        assert np.array_equal(labels.dead, dfl_labels.dead_label(track))
        assert np.array_equal(labels.period, match.period)
        assert track.frame_id.is_unique and all(np.all(np.diff(match.frame[s])==1) for s in halves(match.period))
        timestamp = track.timestamp_s.to_numpy()
        assert all(np.allclose(np.diff(timestamp[s]), .04) for s in halves(match.period))
        conditions = {}
        for condition in ("original","stress"):
            ball = ball_geometry(match,condition,700+MATCHES.index(match_id)*5)
            geom = geometry if condition == "original" else fd.ball_signals(
                features.Match(match.players,match.team,ball,match.period,match.pitch))
            cross = crossings if condition == "original" else fd.line_crossings(geom)
            signals = [movement_features(observed_players[s],match.team,ball[s],match.pitch)
                       for s in halves(match.period)]
            conditions[condition] = (geom, cross, signals)
        data[match_id] = dict(values=values, labels=labels, period=match.period, spells=spells,
                              frame=match.frame, timestamp=timestamp, conditions=conditions,
                              mean_speed=values[:, columns.index("all_speed_mean")],
                              start_kinds=label_start_kinds(spells, events, timestamp))
        print(f"{match_id} 정렬 확인 · {len(track):,}프레임", flush=True)
    return train, data, columns

## 3. 데드볼 보정

기본 확률에 상태별 비용을 더하고, 시작·재개 프레임의 전환 비용을 따로 조정한다.

In [ ]:
def aligned_baseline(score, game, condition, penalty=PENALTY):
    geo,cross,_ = game["conditions"][condition]
    free = np.zeros(len(score),dtype=bool)
    free[cross] = True
    period = game["period"]
    state = smoothing.viterbi(score,period,penalty,BIAS,free_entry=free)
    final,_ = fd.force_line_outs(state,cross,geo,score,period)
    return final.astype(bool)


def viterbi_cost(log_odds, enter, leave):
    # Same two-state recurrence and tie handling as src/smoothing.py, with directed costs.
    alive, dead = 0., float(-log_odds[0])
    back_a = np.zeros(len(log_odds),dtype=bool)
    back_d = np.zeros(len(log_odds),dtype=bool)
    for i in range(1,len(log_odds)):
        switch_a = dead + leave[i]
        switch_d = alive + enter[i]
        back_a[i] = switch_a < alive
        back_d[i] = switch_d < dead
        alive, dead = min(alive,switch_a), min(dead,switch_d) - log_odds[i]
    state = np.empty(len(log_odds),dtype=bool)
    current = dead < alive
    for i in range(len(state)-1,-1,-1):
        state[i] = current
        current = not current if (back_d[i] if current else back_a[i]) else current
    return state


def boundary_input(score, game, condition):
    halves_x = []
    for half, sig in zip(halves(game["period"]),game["conditions"][condition][2]):
        p = score[half]
        log_odds = logit(np.clip(p,1e-6,1-1e-6))
        cols = [log_odds, shift(log_odds,13),shift(log_odds,-13),shift(log_odds,5),shift(log_odds,-5)]
        cols += [sig[name] for name in ("decel","local","fallback")]
        cols += [sig[name] for name in ("restart","kick","throw","pass","carry","stop","prep")]
        for margin in (.0,.25,.50):
            marker = np.zeros(len(p), dtype=bool)
            marker[sig[f"cross_{margin}"]] = True
            cols.append(spread(marker,4))
        halves_x.append(np.column_stack(cols).astype(np.float32))
    return np.concatenate(halves_x)


def boundary_targets(game):
    label = game["labels"].dead.to_numpy().astype(bool)
    out = np.zeros((len(label),2),dtype=bool)
    valid = np.ones(len(label),dtype=bool)
    for half in halves(game["period"]):
        y = label[half]
        changes = np.flatnonzero(y[1:] != y[:-1])+1
        for start in (True,False):
            mask = np.zeros(len(y),dtype=bool)
            mask[changes[y[changes]==start]] = True
            out[half,0 if start else 1] = spread(mask,4)
        valid[half.start:half.start+5] = False
        valid[half.stop-5:half.stop] = False
    return out,valid


def decode(score, game, condition, cases, setting, boundary=None, global_penalty=None):
    if not cases:
        return aligned_baseline(score,game,condition,global_penalty if global_penalty is not None else PENALTY), {}
    geom,cross,signals = game["conditions"][condition]
    result = np.zeros(len(score),dtype=bool)
    support = {name:np.zeros(len(score),dtype=bool) for name in
               ("legacy_cross","line_entry","line_snap","restart","restart_fallback","preparation",
                "pass","carry","keeper","back_pass","short_pair","learned_stop","learned_restart","legacy_changed")}
    strength, bonus = setting["strength"],setting["bonus"]
    for half,sig in zip(halves(game["period"]),signals):
        p = score[half]
        n = len(p)
        old = cross[(cross>=half.start)&(cross<half.stop)]-half.start
        if not sig["reliable"].any():
            result[half] = smoothing.viterbi(p, np.zeros(n), PENALTY,BIAS).astype(bool)
            continue
        log_odds = logit(np.clip(p,1e-6,1-1e-6))
        enter = np.full(n,PENALTY)
        leave = np.full(n,PENALTY)
        enter[old] = 0
        support["legacy_cross"][half.start+old] = True
        restart_ids = np.flatnonzero(sig["restart"])
        candidates = sig[f"cross_{setting['margin']}"]
        certified, short_pairs = [], []
        for a in candidates:
            rs = restart_ids[(restart_ids>a+round(.32*FPS)) & (restart_ids<a+25*FPS)]
            evidence = p[a:min(n,a+2*FPS)]
            outside = sig["distance"][a:min(n,a+8)] < -setting["margin"]
            if len(rs) and (outside.sum()>=4) and (evidence.max()>=.40 or np.any(old==a)):
                b = int(rs[0])
                certified.append((int(a),b))
                if b-a < 10*FPS:
                    short_pairs.append((int(a),b))
        confirmed = np.unique(np.r_[old,np.array([a for a,_ in certified],dtype=int)])
        if 1 in cases:
            picks = np.array([a for a,_ in certified],dtype=int)
            enter[picks] *= (1-.9*strength)
            support["line_entry"][half.start+picks] = True
            stop = sig["stop"]
            enter[stop] *= (1-.55*strength)
            support["line_entry"][half] |= stop
        if 2 in cases:
            snap = extend_from(confirmed,n,round(.72*FPS),0)
            snap[confirmed] = False
            slow_lead = snap & (log_odds<2.0)
            log_odds[slow_lead] -= bonus
            enter[snap] += 15*strength
            support["line_snap"][half] = snap
        context = maximum_filter1d(p, 2*FPS+1, mode="nearest") >= .5
        restart = sig["restart"] & context
        if 3 in cases:
            leave[restart] *= 1-.86*strength
            start_ids = np.flatnonzero(restart)
            after = extend_from(start_ids,n,0,5)
            log_odds[after & sig["reliable"]] -= bonus
            fallback = sig["fallback"] & (p<.58) & context
            leave[fallback] *= 1-.26*strength
            support["restart"][half] |= restart
            support["restart_fallback"][half] = fallback
        if 4 in cases:
            prep = sig["prep"] & context & (p>.22)
            leave[prep] += 22*strength
            log_odds[prep] += .3*bonus
            leave[restart] *= 1-.52*strength
            support["preparation"][half] = prep
            support["restart"][half] |= restart
        if 5 in cases:
            protected = sig["pass"] | sig["carry"]
            # Keep clear line-out entry above competing in-play cues.
            protected = protected & ~extend_from(confirmed,n,2,4)
            enter[protected] += 28*strength
            log_odds[protected] -= 2.5*bonus
            for name in ("pass","carry","keeper","back_pass"):
                support[name][half] = sig[name] & protected
        if 6 in cases:
            for a,b in short_pairs:
                enter[a] = min(enter[a],2.5)
                leave[b] = min(leave[b],2.5)
                log_odds[a:b] += .45*bonus
                support["short_pair"][half.start+a:half.start+b] = True
        if boundary is not None:
            pred = boundary[half]
            real = sig["reliable"] | (sig["decel"]>.28) | sig["fallback"]
            for i, costs, key in ((0,enter,"learned_stop"),(1,leave,"learned_restart")):
                signal = (pred[:,i]>=setting["boundary_cut"]) & real
                costs[signal] *= 1-.68*strength
                support[key][half] = signal
        enter[old] = 0
        result[half] = viterbi_cost(log_odds,enter,leave)
    # Preserve original confirmed line-outs. A corroborated restart may release the lock.
    forced,_ = fd.force_line_outs(result,cross,geom,score,game["period"])
    for half in halves(game["period"]):
        # Restore a release only when the new decoder already switched to ALIVE.
        if 3 in cases or 4 in cases or 6 in cases:
            release = support["restart"][half] | support["learned_restart"][half]
            release_ids = np.flatnonzero(release)
            for a,b in features.runs_of(forced[half] & ~result[half]):
                chosen = release_ids[(release_ids>=max(0,a-3))&(release_ids<b)]
                if len(chosen) and not np.any(support["legacy_cross"][half][max(a,chosen[0]):b]):
                    forced[half.start+max(a,chosen[0]):half.start+b] = False
    support["legacy_changed"] = forced != result
    return forced.astype(bool),support

## 4. 학습과 내부 검증

한 경기를 바깥 평가로 고정한다. 나머지 여섯 경기로 학습하며, 그 안의 한 경기로 설정을 고른다.

In [ ]:
def fit_base(train, columns, members, split=None, value=None):
    frames = [pd.DataFrame(train[m][0],columns=columns).assign(dead=train[m][1]) for m in members]
    saved_stride, saved_params = CONFIG["train_stride"], CONFIG["lgbm_params"]
    params = dict(saved_params,num_threads=THREADS)
    try:
        CONFIG["train_stride"] = 1
        CONFIG["lgbm_params"] = params
        if split == "feature_contri":
            params["feature_contri"] = [value if c.split("@")[0]=="all_speed_p75" else 1.0 for c in columns]
        if split == "forced_split":
            with TemporaryDirectory(prefix="dfl_forced_split_") as folder:
                file = Path(folder) / "splits.json"
                file.write_text(json.dumps(dict(feature=columns.index("all_speed_p75"),threshold=value,
                                                left={},right={})))
                params["forcedsplits_filename"] = str(file)
                model,_ = train_dfl.train(frames)
        else:
            model,_ = train_dfl.train(frames)
    finally:
        CONFIG["train_stride"], CONFIG["lgbm_params"] = saved_stride, saved_params
    return model


def split_stats(model, columns, split, value):
    if split == "forced_split":
        col = columns.index("all_speed_p75")
        roots = [t["tree_structure"] for t in model.dump_model()["tree_info"]]
        thresholds = [r["threshold"] for r in roots if r.get("split_feature")==col]
        hits = [x for x in thresholds if abs(x-value)<1e-9]
        close = [x for x in thresholds if abs(x-value)<.05]
        return dict(root_count=len(roots),matched_root_count=len(hits),near_root_count=len(close),
                    target_threshold=value, actual_root_threshold_counts=dict(sorted(
                        ((str(x),thresholds.count(x)) for x in set(thresholds)), key=lambda pair:-pair[1])[:8]))
    gain = model.feature_importance("gain")
    used = np.array([c.split("@")[0]=="all_speed_p75" for c in columns])
    return dict(speed_p75_gain_share=float(gain[used].sum()/gain.sum()),gain_multiplier=value)


def predictions(train, games, columns):
    # Cache only out-of-fold probabilities in memory, with their exact training membership.
    scores = {}
    def get(test, exclude=()):
        excluded = frozenset((test,*exclude))
        key = test, excluded
        if key not in scores:
            members = tuple(m for m in MATCHES if m not in excluded)
            assert test not in members
            model = fit_base(train, columns,members)
            for other in excluded:
                scores[other, excluded] = model.predict(games[other]["values"],num_threads=THREADS)
        return scores[key]
    return get


def fit_boundaries(games, score_of, members):
    xx, yy = [], []
    for m in members:
        targets, valid = boundary_targets(games[m])
        rows = np.arange(len(valid))
        use = valid & (targets.any(axis=1) | (rows%7==0))
        for cond in ("original","stress"):
            xx.append(boundary_input(score_of(m),games[m],cond)[use])
            yy.append(targets[use])
    x, y = np.concatenate(xx),np.concatenate(yy)
    del xx,yy
    models = []
    for i in range(2):
        weight = np.where(y[:,i], 5.0, 1.0)
        models.append(lgb.train(dict(objective="binary",learning_rate=.07,num_leaves=23,
                                     min_data_in_leaf=180,num_threads=THREADS,verbosity=-1,seed=17),
                                lgb.Dataset(x,label=y[:,i].astype(int),weight=weight),
                                num_boost_round=110))
    return models


def boundary_predictions(models, score, game, condition):
    x = boundary_input(score,game,condition)
    return np.column_stack([m.predict(x,num_threads=THREADS) for m in models])


def objective(game, states):
    scores = [scoring.score_one(pred,game["labels"]) for pred in states]
    spell = np.mean([s["alive_spell_f1"] for s in scores])
    frame = np.mean([s["dead_f1"] for s in scores])
    return float(spell + .20*frame)


def select_setting(score, game, variant, boundary):
    cases = VARIANTS[variant]
    learned = variant in ("6_short_missed","full_combined")
    settings = [dict(s,boundary_cut=cut) for s in RULE_GRID for cut in
                ((.48,.64) if learned else (.64,))]
    if not any(c in cases for c in (1,2,6)):
        settings = [s for s in settings if s["margin"]==.25]
    ranked = []
    for setting in settings:
        states = [decode(score,game,cond,cases,setting,boundary.get(cond) if learned else None)[0]
                  for cond in ("original","stress")]
        ranked.append(objective(game,states))
    return settings[int(np.argmax(ranked))]


def choose_split(train,games,columns,inner,members,split):
    candidates = P75_WEIGHTS if split=="feature_contri" else FORCED_THRESHOLDS
    ranked = []
    for value in candidates:
        model = fit_base(train, columns,members,split,value)
        score = model.predict(games[inner]["values"],num_threads=THREADS)
        states = [aligned_baseline(score,games[inner],cond) for cond in ("original","stress")]
        ranked.append(objective(games[inner],states))
    return candidates[int(np.argmax(ranked))]

## 5. 오류 평가

경계가 0.5초를 넘게 어긋난 구간, 누락한 짧은 중단, 인플레이 오탐을 센다.

In [ ]:
CASE_NAMES = {1:"start_late", 2:"start_early", 3:"alive_late", 4:"alive_early",
              5:"false_dead", 6:"short_missed"}
CASE_STRATA = {1:("throw_in","goal_kick","corner","foul","offside"),
               2:("throw_in","goal_kick","corner"),
               3:("throw_in","goal_kick","free_kick"),
               4:("throw_in","goal_kick","free_kick","corner"),
               5:("low_tempo","back_build_up","keeper_candidate"),
               6:("throw_in","goal_kick","free_kick","offside")}


def case_masks(state, game, condition):
    label = game["labels"].dead.to_numpy(dtype=bool)
    spells = game["spells"]
    marks = {c:{} for c in CASE_NAMES}
    hit = np.zeros(len(spells),dtype=bool)
    start_errors, end_errors = [], []
    for hs,sig in zip(halves(game["period"]),game["conditions"][condition][2]):
        y = label[hs]
        pred = state[hs]
        ts = [(a,b) for a,b in features.runs_of(y)]
        ps = [(a,b) for a,b in features.runs_of(pred)]
        gt = spells[spells.period==game["period"][hs.start]]
        assert len(ts)==len(gt)
        for ix,(a,b) in zip(gt.index,ts):
            hit[ix] = pred[a:b].mean()>=.5
            cause = gt.loc[ix,"cause"]
            start_kind = game["start_kinds"].get(hs.start+a,cause)
            if not hit[ix] and b-a<10*FPS:
                marks[6][(hs.start+a,hs.start+b)] = "offside" if start_kind=="offside" else cause
            if not hit[ix]:
                continue
            overlaps = [(x,z) for x,z in ps if x<b and z>a]
            if len(overlaps)!=1:
                continue
            x,z = overlaps[0]
            if sum(u<z and v>x for u,v in ts)!=1:
                continue
            if a and x:
                start_errors.append(abs(x-a)/FPS)
                if abs(x-a)>.5*FPS:
                    span = (hs.start+min(x,a),hs.start+max(x,a))
                    marks[1 if x>a else 2][span] = start_kind if start_kind in ("foul","offside") else cause
            if b<len(y) and z<len(y):
                end_errors.append(abs(z-b)/FPS)
                if abs(z-b)>.5*FPS:
                    span = (hs.start+min(z,b),hs.start+max(z,b))
                    marks[3 if z>b else 4][span] = cause
        for a,b in ps:
            if y[a:b].mean()<.5:
                if sig["keeper"][a:b].any():
                    kind = "keeper_candidate"
                elif sig["back_pass"][a:b].any():
                    kind = "back_build_up"
                else:
                    kind = "low_tempo" if np.nanmedian(game["mean_speed"][hs][a:b])<2 else "other"
                marks[5][(hs.start+a,hs.start+b)] = kind
    durations = spells.seconds.to_numpy()
    short3,short10 = durations<3,durations<10
    stats = dict(short3_n=int(short3.sum()), short3_found=int(hit[short3].sum()),
                 short10_n=int(short10.sum()), short10_found=int(hit[short10].sum()),
                 false_dead=len(marks[5]), missed_dead=int((~hit).sum()),
                 start_mae=float(np.mean(start_errors)),end_mae=float(np.mean(end_errors)))
    return marks,stats


def match_errors(old, new):
    # One-to-one match on temporal overlap. Partial fragments count as new only when unmatched.
    pairs = []
    for i,(a,b) in enumerate(old):
        for j,(c,d) in enumerate(new):
            overlap = max(0,min(b,d)-max(a,c))
            if overlap:
                pairs.append((overlap/(max(b,d)-min(a,c)),i,j))
    left,right = set(),set()
    for overlap,i,j in sorted(pairs,reverse=True):
        if i not in left and j not in right:
            left.add(i)
            right.add(j)
    return ([s for i,s in enumerate(old) if i not in left],
            [s for j,s in enumerate(new) if j not in right])


def compare_errors(match, cond, variant, game, old, new, original, corrected, support):
    rows, clips = [], []
    label = game["labels"].dead.to_numpy(dtype=bool)
    for i,name in CASE_NAMES.items():
        a,b = old[i],new[i]
        for kind in ("all",*CASE_STRATA[i]):
            keys = lambda t: [span for span,cause in t.items() if kind=="all" or kind==cause]
            baseline_spans,new_spans = keys(a),keys(b)
            fixed,introduced = match_errors(baseline_spans,new_spans)
            rows.append(dict(match=match,condition=cond,variant=variant,case=i,error=name,
                             stratum=kind,baseline=len(baseline_spans),count=len(new_spans),
                             fixed=len(fixed),introduced=len(introduced)))
            if kind!="all":
                continue
            for change,segments,source in (("fixed",fixed,a),("introduced",introduced,b)):
                for start,stop in segments:
                    sl=slice(start,stop)
                    majority=lambda v: "D" if v[sl].mean()>=.5 else "ALIVE"
                    signal = "|".join(key for key,mask in support.items() if mask[sl].any())
                    if not signal and support:
                        nearby = slice(max(0,start-FPS),min(len(label),stop+FPS))
                        signal = "|".join(key for key,mask in support.items() if mask[nearby].any())
                    clips.append(dict(match=match,condition=cond,variant=variant,case=i,stratum=source[(start,stop)],
                                      change=change,half=int(game["period"][start]),
                                      start_frame=int(game["frame"][start]),last_frame=int(game["frame"][stop-1]),
                                      start_s=float(game["timestamp"][start]),end_s=float(game["timestamp"][stop-1]+1/FPS),
                                      gt=majority(label),base=majority(original),final=majority(corrected),
                                      gt_dead_share=float(label[sl].mean()),base_dead_share=float(original[sl].mean()),
                                      final_dead_share=float(corrected[sl].mean()),signals=signal))
    return rows,clips

## 6. DFL 실행과 결과 저장

인플레이 구간 F1, 데드볼 구간 F1, 데드볼 프레임 F1을 각각 기록한다.

In [ ]:
def summarize(rows):
    groups = rows.groupby(["condition","variant"],sort=False)
    out = groups.agg(matches=("match","size"),alive_spell_f1=("alive_spell_f1","mean"),
                     dead_spell_f1=("dead_spell_f1","mean"),
                     frame_f1=("dead_f1","mean"),short3_n=("short3_n","sum"),short3_found=("short3_found","sum"),
                     short10_n=("short10_n","sum"),short10_found=("short10_found","sum"),
                     false_dead=("false_dead","sum"),missed_dead=("missed_dead","sum"),
                     start_mae=("start_mae","mean"),end_mae=("end_mae","mean"),
                     inference_s=("inference_s","sum"),shared_fit_s=("shared_fit_s","sum"),
                     variant_fit_s=("variant_fit_s","sum")).reset_index()
    for n in (3,10):
        out[f"short{n}_recall"] = out[f"short{n}_found"] / out[f"short{n}_n"]
    return out


def json_file(path, payload):
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(payload,ensure_ascii=False,indent=2)+"\n")
    temp.replace(path)


def frame_file(path, frame):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp,index=False)
    temp.replace(path)


def input_hashes():
    paths = [NOTEBOOK, ROOT / "evaluation/possession_metrics.py"]
    paths += [ROOT / f"src/{p}.py" for p in ("config", "train_dfl", "features", "simulate", "smoothing",
                                          "dfl_load", "dfl_labels", "final_dead_balls", "compare_smoothing")]
    cache = Path(train_dfl.feature_dir())
    for match in MATCHES:
        paths.extend(Path(p) for p in dfl_load.cache_paths(match))
        paths.extend((cache / f"{match}_rand_{TRAIN_SEEDS[match]}.parquet",
                      cache / f"{match}_sim_{SEED}.parquet"))
    return {str(path.relative_to(ROOT)): sha(path) for path in paths}


def score_prediction(state, labels):
    scores = scoring.score_one(state, labels)
    period = labels.period.to_numpy()
    # 기존 구간 채점기는 H만 매칭하므로, D를 평가할 때 D 프레임을 H로 매핑한다.
    gt = scoring.codes_of(~labels.dead.to_numpy(dtype=bool), period)
    pred = scoring.codes_of(~state, period)
    spell = scoring.pm.spell_scores(gt, pred, scoring.pm.valid_runs(gt))
    return dict(scores, dead_spell_f1=spell["mean_f1"],
                dead_spell_f1_iou50=spell["per_iou"][.5]["f1"])


def run(matches=MATCHES):
    start = perf_counter()
    train, games, columns = prepare_data()
    hashes = input_hashes()
    prepare_s = perf_counter()-start
    get_score = predictions(train,games,columns)
    rows, errors, clips, folds = [], [], [], []
    for held in matches:
        tic = perf_counter()
        print(f"{held} · 기준 확률 및 내부 검증 준비",flush=True)
        game = games[held]
        outer = [m for m in MATCHES if m!=held]
        inner = INNER[held]
        inner_train = [m for m in outer if m!=inner]
        base_held = get_score(held)
        old, old_masks = {}, {}
        for cond in ("original","stress"):
            old[cond] = aligned_baseline(base_held,game,cond)
            old_masks[cond], stats = case_masks(old[cond],game,cond)
        inner_score = get_score(inner,(held,))
        print(f"{held} · 내부 검증 {inner}, 보정 모델 학습",flush=True)
        inner_models = fit_boundaries(games,lambda m:get_score(m,(held,inner)),inner_train)
        inner_prob = {c:boundary_predictions(inner_models,inner_score,games[inner],c) for c in ("original","stress")}
        del inner_models
        outer_models = fit_boundaries(games,lambda m:get_score(m,(held,)),outer)
        outer_prob = {c:boundary_predictions(outer_models,base_held,game,c) for c in ("original","stress")}
        del outer_models
        shared_fit_s = perf_counter()-tic
        settings = {}
        times = {}
        split_scores = {}
        for name in VARIANTS:
            if name=="baseline":
                continue
            tick = perf_counter()
            settings[name] = select_setting(inner_score,games[inner],name,inner_prob)
            times[name] = perf_counter()-tick
        tick = perf_counter()
        penalties = (15.,25.)
        penalty = max(penalties,key=lambda v:objective(games[inner],[
            aligned_baseline(inner_score,games[inner],c,v) for c in ("original","stress")]))
        settings["global_penalty"] = dict(penalty=penalty)
        times["global_penalty"] = perf_counter()-tick
        for name in SPLIT_VARIANTS:
            tick = perf_counter()
            value = choose_split(train,games,columns,inner,inner_train,name)
            model = fit_base(train, columns,outer,name,value)
            split_scores[name] = model.predict(game["values"],num_threads=THREADS)
            settings[name] = split_stats(model,columns,name,value)
            times[name] = perf_counter()-tick

        masked = dict(game,conditions=dict(game["conditions"]))
        masked_signals = [{"reliable":np.zeros(half.stop-half.start,dtype=bool)} for half in halves(game["period"])]
        blank = {"inside":np.zeros(len(base_held),dtype=bool)}
        masked["conditions"]["masked"] = (blank,np.array([],dtype=int),masked_signals)
        masked_base,_ = decode(base_held,masked,"masked",(),{})
        masked_full,_ = decode(base_held,masked,"masked",VARIANTS["full_combined"],
                               settings["full_combined"],np.zeros((len(base_held),2)))
        np.testing.assert_array_equal(masked_base,masked_full)

        condition_print = {}
        for cond in ("original","stress"):
            for name in (*VARIANTS,"global_penalty",*SPLIT_VARIANTS):
                tick = perf_counter()
                if name=="baseline":
                    pred,support = old[cond],{}
                elif name=="global_penalty":
                    pred,support = decode(base_held,game,cond,(),{},global_penalty=penalty)
                elif name in SPLIT_VARIANTS:
                    pred,support = decode(split_scores[name],game,cond,(),{})
                else:
                    learned = name in ("6_short_missed","full_combined")
                    pred,support = decode(base_held,game,cond,VARIANTS[name],settings[name],
                                          outer_prob[cond] if learned else None)
                inference_s = perf_counter()-tick
                assert pred.shape==base_held.shape and pred.dtype==bool
                labels = game["labels"]
                score = score_prediction(pred,labels)
                marks,stats = case_masks(pred,game,cond)
                by_case,new_clips = compare_errors(held,cond,name,game,old_masks[cond],marks,
                                                    old[cond],pred,support)
                errors.extend(by_case)
                clips.extend(new_clips)
                rows.append(dict(match=held,condition=cond,variant=name,inference_s=inference_s,
                                 shared_fit_s=shared_fit_s,variant_fit_s=times.get(name,0.),
                                 **score,**stats))
                if name=="full_combined":
                    condition_print[cond] = score
        folds.append(dict(held_match=held,outer_train=outer,inner_validation=inner,inner_train=inner_train,
                          settings=settings,shared_fit_s=shared_fit_s,wall_s=perf_counter()-tic,
                          stress_observed_share=float(game["conditions"]["stress"][0]["seen"].mean()),
                          masked_fallback_exact=True))
        a,b = condition_print["original"],condition_print["stress"]
        print(f"{held} 완료 · 원본 D Spell {a['dead_spell_f1']:.4f} / ALIVE Spell {a['alive_spell_f1']:.4f} / D 프레임 {a['dead_f1']:.4f}"
              f" · 공 노이즈 D Spell {b['dead_spell_f1']:.4f} / ALIVE Spell {b['alive_spell_f1']:.4f} / D 프레임 {b['dead_f1']:.4f}"
              f" · {folds[-1]['wall_s']:.0f}초",flush=True)
    return pd.DataFrame(rows),pd.DataFrame(errors),pd.DataFrame(clips),folds,hashes,prepare_s,perf_counter()-start


def write_report(output, table, wall):
    lines = [
        "# DFL LightGBM v2 데드볼 보정 결과", "",
        f"DFL 7경기 · 6경기 학습/1경기 평가 · 전체 실행 {wall / 60:.1f}분", "",
        "| 공 트랙 | 조건 | 데드볼 구간 F1 | 인플레이 구간 F1 | 데드볼 프레임 F1 | 3초 미만 재현율 | 10초 미만 재현율 | 인플레이 오탐 | 시작 오차(초) | 재개 오차(초) |",
        "|---|---|---:|---:|---:|---:|---:|---:|---:|---:|",
    ]
    for row in table.itertuples(index=False):
        lines.append(f"| {row.condition} | {row.variant} | {row.dead_spell_f1:.4f} | "
                     f"{row.alive_spell_f1:.4f} | {row.frame_f1:.4f} | {row.short3_recall:.1%} | "
                     f"{row.short10_recall:.1%} | {row.false_dead} | {row.start_mae:.2f} | {row.end_mae:.2f} |")
    lines.extend(["", "경기별 점수와 시간은 [per_match.csv](per_match.csv), "
                  "오류 유형별 변화는 [error_by_case.csv](error_by_case.csv)에서 확인할 수 있다."])
    (output / "RESULTS.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def write_results(run_result, output):
    rows, errors, clips, folds, hashes, preparation, wall = run_result
    table = summarize(rows)
    assert len(folds)==7 and set(rows.match)==set(MATCHES)
    assert len(rows)==7*2*(len(VARIANTS)+len(SPLIT_VARIANTS)+1)
    assert not rows.duplicated(["match","condition","variant"]).any()
    assert rows[["alive_spell_f1","dead_spell_f1","dead_f1"]].stack().between(0,1).all()
    assert ((errors.baseline-errors.fixed)==(errors["count"]-errors.introduced)).all()
    assert (clips.last_frame>=clips.start_frame).all()
    assert all(f["masked_fallback_exact"] for f in folds)
    assert all(f["held_match"] not in f["outer_train"] and f["inner_validation"] not in f["inner_train"] for f in folds)
    output.mkdir(parents=True,exist_ok=False)
    for name, data in (("comparison",table),("per_match",rows),("error_by_case",errors),("error_clips",clips)):
        frame_file(output/f"{name}.csv",data)
    sha_out = {p.name:sha(p) for p in output.glob("*.csv")}
    command = subprocess.run(["git","rev-parse","HEAD"],cwd=ROOT,capture_output=True,text=True)
    json_file(output/"run_manifest.json",dict(run_id=output.name,fps=FPS,matches=folds,preparation_s=preparation,
              wall_s=wall,git_commit=command.stdout.strip(),packages={n:version(n) for n in
              ("lightgbm","numpy","scipy","pandas","pyarrow")},stress=STRESS,
              rule_grid=RULE_GRID,variants=VARIANTS,input_sha256=hashes,output_sha256=sha_out))
    baseline_counts = errors.query("condition=='original' and variant=='baseline' and stratum=='all'")
    counts_match = baseline_counts.groupby("case")["count"].sum().tolist() == BASELINE_ERRORS
    baseline = table.query("condition == 'original' and variant == 'baseline'").iloc[0]
    f1_matches = (round(baseline.alive_spell_f1, 4) == BASELINE_F1["alive_spell_f1"]
                  and round(baseline.frame_f1, 4) == BASELINE_F1["dead_f1"])
    json_file(output/"validation.json",dict(passed=True,matches=len(folds),metric_rows=len(rows),
              variant_count=rows.variant.nunique(),case_rows=len(errors),clip_rows=len(clips),
              baseline_f1_matches_reference=bool(f1_matches),baseline_error_counts_exact=counts_match,
              masked_fallback_exact=True,
              all_requested_case_variants_and_combinations=True))
    write_report(output, table, wall)
    return table

## 7. BAR–ATH 적용

DFL 7경기로 학습한 가중치와 영상에 적용한 규칙 결합을 사용한다.

In [ ]:
def predict_competition():
    """BAR–ATH 전술 좌표에 영상에서 사용한 ①~⑥ 규칙 결합을 적용한다."""
    import competition_data

    match, _ = competition_data.load_competition_match()
    graded = match.period > 0
    comp = features.Match(players=match.players[graded], team=match.team, ball=match.ball[graded],
                          period=match.period[graded], pitch=match.pitch, frame=match.frame[graded])
    model = lgb.Booster(model_file=train_dfl.model_path("v2"))
    matrix = features.compute_features(comp)[model.feature_name()].astype(np.float32)
    probability = model.predict(matrix, num_threads=THREADS)
    del matrix
    geometry = fd.ball_signals(comp)
    crossings = fd.line_crossings(geometry)
    crossings = crossings[(crossings > 0) & (comp.period[crossings] == comp.period[crossings - 1])]
    signals = [movement_features(comp.players[h], comp.team, comp.ball[h], comp.pitch) for h in halves(comp.period)]
    game = dict(period=comp.period, conditions={"original": (geometry, crossings, signals)})
    state, _ = decode(probability, game, "original", VARIANTS["rules_combined"], COMPETITION_RULES)

    _, second_start, second_kickoff = CONFIG["final"]["public_kickoffs"]
    pre_kickoff = (comp.frame >= second_start) & (comp.frame < second_kickoff)
    state[pre_kickoff] = True
    return pd.DataFrame({"frame": comp.frame, "half": comp.period,
                         "dead_probability": probability, "state": np.where(state, "D", "ALIVE")})

## 8. 실행 방식 선택

아래의 `RUN_MODE`를 바꾼 뒤 **Run All**을 누른다. 기존 실행 폴더를 유지하려면 `DFL_CORRECTIONS_RUN_ID`에 새 폴더 이름을 지정한다.

In [ ]:
# "full"은 7경기 평가, "smoke"는 첫 평가 경기, "competition"은 BAR–ATH에 적용.
RUN_MODE = "full"

if __name__ == "__main__":
    mode = os.environ.get("DFL_CORRECTIONS_MODE", RUN_MODE)
    run_id = os.environ.get("DFL_CORRECTIONS_RUN_ID", f"lightgbm_corrections_{mode}")
    if not run_id or Path(run_id).name != run_id or run_id in (".", ".."):
        raise ValueError("실행 ID는 폴더 이름 하나로 지정해 주세요.")
    output = RESULTS / run_id

    if mode == "full":
        if output.exists():
            raise FileExistsError(f"기존 결과가 있습니다. 새 실행 ID를 지정해 주세요. {output}")
        table = write_results(run(), output)
        print(table.round(4).to_string(index=False), flush=True)
    elif mode == "smoke":
        rows, *_ = run(MATCHES[:1])
        print(summarize(rows).round(4).to_string(index=False), flush=True)
    elif mode == "competition":
        if output.exists():
            raise FileExistsError(f"기존 결과가 있습니다. 새 실행 ID를 지정해 주세요. {output}")
        status = predict_competition()
        output.mkdir(parents=True, exist_ok=False)
        frame_file(output / "frame_status.csv", status)
        print(f"{len(status):,}프레임 · {output / 'frame_status.csv'}", flush=True)
    else:
        raise ValueError(f"지원하는 실행 방식은 full, smoke, competition입니다. 입력값 {mode!r}")